# Module 9 · 圖像 2：影像 → 張量前處理（2026 視覺管線的起點）

> **本節定位｜2026 主流核心**
> 所有視覺模型（ViT、CLIP、ResNet…）吃的都是**規整的浮點張量**，不是 JPG/PNG。
> 把原始影像正確地 decode → resize → normalize → 打包成 batch 張量，
> 是視覺資料前處理**最關鍵的第一步**。

## 學習目標
- 掌握標準前處理四步：**decode → resize/crop → to-tensor → normalize**。
- 釐清張量排列慣例：**PyTorch `(N, C, H, W)` vs TensorFlow/numpy `(N, H, W, C)`**。
- 用 `torchvision.transforms.v2` 與 HuggingFace `AutoImageProcessor` 產出模型輸入。
- 認識不同任務的**標籤資料結構**（分類 / 多標籤 / 偵測 / 分割）。

## 0. 資料結構設計：影像張量與標籤

| 物件 | 形狀 | 型別 | 說明 |
|:--|:--|:--|:--|
| 原始檔 | — | JPG/PNG | 壓縮編碼，**不能直接餵模型** |
| 解碼後像素 | `(H, W, C)` | uint8 [0,255] | PIL/numpy 慣例（通道在最後 = channels-last） |
| 模型輸入（PyTorch） | `(N, C, H, W)` | float32 | **channels-first**，已 normalize |
| 模型輸入（TF/Keras） | `(N, H, W, C)` | float32 | channels-last |

**標籤格式（依任務）**：

| 任務 | 標籤形狀 | 範例 |
|:--|:--|:--|
| 影像分類 | `(N,)` | 類別索引 `3` |
| 多標籤分類 | `(N, num_classes)` | one-hot/multi-hot `[0,1,1,0]` |
| 物件偵測 | 每圖 `list[bbox+類別]` | `[x,y,w,h,cls]` |
| 語意分割 | `(N, H, W)` | 每像素一個類別 |

In [ ]:
import numpy as np
from PIL import Image

# 合成一張 RGB 影像（無需下載）
arr = (np.random.RandomState(0).rand(180, 240, 3) * 255).astype(np.uint8)
pil_img = Image.fromarray(arr)
print(f"原始影像（PIL/numpy 慣例）: {arr.shape}  (H, W, C), dtype={arr.dtype}, 值域[0,255]")

## 1. 標準前處理管線（torchvision.transforms.v2）

`v2` 是 torchvision 現行（2026）推薦的 transforms API。一條 pipeline 把 PIL 影像
變成已 normalize 的 `(C, H, W)` float 張量。需 `uv sync --extra multimodal`。

In [ ]:
try:
    import torch
    from torchvision.transforms import v2

    preprocess = v2.Compose([
        v2.Resize((224, 224)),                 # 統一尺寸
        v2.ToImage(),                          # PIL -> tensor (C, H, W) uint8
        v2.ToDtype(torch.float32, scale=True), # 轉 float 並縮放到 [0,1]
        v2.Normalize(mean=[0.485, 0.456, 0.406],   # ImageNet 統計值
                     std=[0.229, 0.224, 0.225]),
    ])
    x = preprocess(pil_img)
    print(f"單張前處理後: {tuple(x.shape)}  (C, H, W) ← PyTorch channels-first")
    print(f"dtype={x.dtype}, 值域約 [{x.min():.2f}, {x.max():.2f}]（normalize 後可為負）")

    # 打包成 batch
    batch = torch.stack([x, x, x])
    print(f"批次張量: {tuple(batch.shape)}  (N, C, H, W)")
    HAS_TV = True
except Exception as e:
    HAS_TV = False
    print("未安裝 torchvision，請 `uv sync --extra multimodal`。說明仍可閱讀。", e)

**為什麼要 normalize？** 預訓練模型是用 ImageNet 的 mean/std 標準化過的影像訓練的，
推論/微調時必須用**相同的統計值**前處理，否則分佈不一致會嚴重掉分。

## 2. channels-first vs channels-last（最常見的踩雷點）

In [ ]:
if HAS_TV:
    chw = x                                   # (C, H, W) PyTorch
    hwc = x.permute(1, 2, 0)                  # (H, W, C) numpy/TF/繪圖慣例
    print(f"PyTorch 模型輸入   (C, H, W) = {tuple(chw.shape)}")
    print(f"numpy/TF/matplotlib (H, W, C) = {tuple(hwc.shape)}")
    print("→ 用 plt.imshow 顯示要 (H,W,C)；餵 PyTorch 模型要 (C,H,W)。轉換用 permute。")

## 3. HuggingFace AutoImageProcessor：跟著模型走的前處理

和文字的 `AutoTokenizer` 對應，視覺有 `AutoImageProcessor`：載入哪個模型，
就用它**配套**的前處理（正確的尺寸、mean/std），不必自己查。

In [ ]:
if HAS_TV:
    try:
        from transformers import AutoImageProcessor
        proc = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224")
        out = proc(images=pil_img, return_tensors="pt")
        print(f"AutoImageProcessor 輸出 pixel_values: {tuple(out['pixel_values'].shape)}  (N, C, H, W)")
        print("→ 直接得到該 ViT 期望的張量；下一節就把它餵進模型抽特徵。")
    except Exception as e:
        print("（未能下載 ViT 前處理設定，略過）", e)

## 小結

| 重點 | 內容 |
|:--|:--|
| 四步前處理 | decode → resize/crop → to-tensor → normalize |
| PyTorch 張量 | `(N, C, H, W)` float32，channels-first |
| TF/numpy 張量 | `(N, H, W, C)`，channels-last |
| normalize | 用與預訓練一致的 mean/std（如 ImageNet） |
| 跟著模型走 | `AutoImageProcessor` 自動對應正確前處理 |

**下一節 `03_modern_image_representations`**：把 `(N, C, H, W)` 餵進
**預訓練 ViT / CLIP**，得到通用影像特徵（取代舊的 VGG16 手法）。